# Day 20: Mock interview and world-model math

*Week 4: ML from scratch*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## A. Mock interview: merge k sorted lists

`Algorithms`

**Treat this as a real interview.** Set a 25-minute timer, and talk through your approach out loud (or write it as comments) **before** coding.

Write `merge_k_sorted(lists)`: merge k sorted lists into one sorted list. Aim for O(N log k), where N is the total number of elements.

**Look up:** `heapq`, pushing tuples `(value, list_index, element_index)` (why does the tuple need `list_index`?).

**Say out loud afterwards:** why is this better than concatenating and sorting, in O(N log N)? When does that difference actually matter?

In [ ]:
def merge_k_sorted(lists):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import random
assert merge_k_sorted([[1, 4, 5], [1, 3, 4], [2, 6]]) == [1, 1, 2, 3, 4, 4, 5, 6]
assert merge_k_sorted([]) == [] and merge_k_sorted([[], [1]]) == [1]
random.seed(0)
lists = [sorted(random.choices(range(1000), k=random.randint(0, 50))) for _ in range(20)]
assert merge_k_sorted(lists) == sorted(x for l in lists for x in l)
print("✅ A passed")

## B. Word break

`Algorithms`

Write `word_break(s, words)`: return `True` if `s` can be split into a sequence of one or more words from `words` (words can be reused).

```
word_break("applepenapple", ["apple", "pen"])  ->  True
word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])  ->  False
```

Define `dp[i]` = "the first `i` characters can be split". The last test is designed to time out a plain recursive solution with no memoization.

In [ ]:
def word_break(s, words):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
assert word_break("leetcode", ["leet", "code"]) is True
assert word_break("applepenapple", ["apple", "pen"]) is True
assert word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]) is False
assert word_break("", ["a"]) is True
assert word_break("a" * 40 + "b", ["a", "aa", "aaa", "aaaa"]) is False
print("✅ B passed")

## C. World-model math: reparameterization, KL, and the MDN loss

`ML from scratch`

The three formulas behind your V and M models. Write all three in PyTorch, without `torch.distributions` (the tests use it to check you).

1. `reparameterize(mu, logvar)`: z = μ + σ·ε, with ε ~ N(0, I). Gradients must flow back to `mu` and `logvar`.
2. `kl_standard_normal(mu, logvar)`: KL( N(μ, σ²) ‖ N(0, I) ), **summed over the latent dimension**, returning shape (B,). Derive the closed form, or look it up and make sure you understand each term.
3. `mdn_nll(pi_logits, mu, log_sigma, target)`: the negative log-likelihood of `target` under a mixture of K diagonal Gaussians. This is your MDN-RNN's loss.
   - Shapes: `pi_logits` (B, K), `mu` and `log_sigma` (B, K, D), `target` (B, D). Return shape (B,).
   - log p(y) = logsumexp over k of [ log πₖ + Σ_d log N(y_d; μ_kd, σ_kd) ].
   - Use `log_softmax` for log π, and **logsumexp** for the sum over components. Never compute probabilities and then take a log.

**Say out loud:** why must the MDN loss use logsumexp? What happens to a naive version when D = 32?

In [ ]:
def reparameterize(mu, logvar):
    # your code here
    pass


def kl_standard_normal(mu, logvar):
    # your code here
    pass


def mdn_nll(pi_logits, mu, log_sigma, target):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
from torch.distributions import Normal, Categorical, Independent, MixtureSameFamily, kl_divergence
torch.manual_seed(0)
mu = torch.tensor([1.0, -2.0]).repeat(200_000, 1).requires_grad_()
logvar = torch.log(torch.tensor([4.0, 0.25])).repeat(200_000, 1).requires_grad_()
z = reparameterize(mu, logvar)
assert torch.allclose(z.mean(0), torch.tensor([1.0, -2.0]), atol=0.02)
assert torch.allclose(z.std(0), torch.tensor([2.0, 0.5]), atol=0.02)
z.sum().backward()
assert mu.grad is not None and logvar.grad is not None, "gradients must reach mu and logvar"
mu, logvar = torch.randn(6, 4), torch.randn(6, 4)
kl = kl_standard_normal(mu, logvar)
expected = kl_divergence(Normal(mu, torch.exp(0.5 * logvar)), Normal(0.0, 1.0)).sum(-1)
assert kl.shape == (6,) and torch.allclose(kl, expected, atol=1e-5)
B, K, D = 5, 3, 32
pi_logits, m, ls, y = torch.randn(B, K), torch.randn(B, K, D), torch.randn(B, K, D) * 0.3, torch.randn(B, D)
mix = MixtureSameFamily(Categorical(logits=pi_logits), Independent(Normal(m, ls.exp()), 1))
nll = mdn_nll(pi_logits, m, ls, y)
assert nll.shape == (B,) and torch.allclose(nll, -mix.log_prob(y), atol=1e-4)
far = mdn_nll(pi_logits, m, ls, y + 50)
assert torch.all(torch.isfinite(far)), "far-away targets give inf: use logsumexp"
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.